# Tahap 1 — U-Net Beku & Uji Kualitas Batas Paru (RSUA & Kermany)

Ketentuan:
1. **Pemuatan Bobot U-Net Beku:** Memuat bobot pralatih publik (Zenodo Record 8393556, dilatih pada Darwin + Montgomery + Shenzhen) dengan parameter yang dikunci sepenuhnya (`requires_grad = False`).
2. **Pemeriksaan Kuantitatif Pendukung pada Dataset RSUA:** Mengukur nilai *Dice Similarity Coefficient* (DSC) dan *Intersection over Union* (IoU) terhadap penanda batas paru dokter spesialis radiologi pada 85 citra (32 Normal dan 53 Pneumonia). Target kriteria proposal: $\text{DSC} \ge 0{,}85$ dan $\text{IoU} \ge 0{,}75$.
3. **Pemeriksaan pada Citra Anak Kermany:** Memeriksa sebaran rasio luas paru ($r_{\text{paru}}$) pada data uji Kermany (minimal 50 citra: 25 Normal, 25 Pneumonia), mendeteksi kegagalan ekstrem ($r_{\text{paru}} < 0{,}05$), serta menguji selisih rata-rata antarkelas ($|\Delta \bar{r}_{\text{paru}}| > 0{,}05$) untuk menentukan kewajiban pelaporan $\text{ACR}_{\text{norm}}$.
4. **Visualisasi Kontur:** Mengekspor visualisasi garis batas paru (*contour overlay*) pada citra rontgen asli untuk verifikasi visual.

In [ ]:
# Sel 1: Inisialisasi lingkungan Colab, instal dependensi segmentasi, dan resolusi path
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive  # type: ignore
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    !pip install -q segmentation-models-pytorch==0.6.0 opencv-python-headless==4.11.0.86

for candidate in [Path.cwd(), Path.cwd().parent, Path("/content/drive/MyDrive/LAB_TA_GradCAM_Pneumonia/code")]:
    if (candidate / "src" / "paths.py").exists():
        sys.path.insert(0, str(candidate))
        break

import torch
from src.paths import get_project_paths, ensure_output_dirs, load_config

CFG = load_config()
PATHS = ensure_output_dirs(cfg=CFG)
PROJECT_ROOT = PATHS["project_root"]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"IN_COLAB     : {IN_COLAB}")
print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"DEVICE       : {DEVICE} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

### Analisis Sel 1
Memastikan pustaka `segmentation_models_pytorch` terpasang dan perangkat GPU CUDA aktif untuk inferensi cepat U-Net.

In [ ]:
# Sel 2: Unduh dan muat bobot model U-Net beku (Zenodo 8393556)
from scripts.download_unet_weights import download_and_extract_unet_weights
from src.segmentation.unet import load_frozen_unet

weights_dir = PATHS["unet_frozen"]
weights_path = download_and_extract_unet_weights(target_dir=weights_dir, force_download=False)

print("\nMemuat model U-Net beku ke memori komputasi...")
unet_model = load_frozen_unet(weights_path, device=DEVICE)

# Verifikasi matematis bahwa 100% parameter beku (requires_grad = False)
total_params = sum(p.numel() for p in unet_model.parameters())
trainable_params = sum(p.numel() for p in unet_model.parameters() if p.requires_grad)

print(f"Total Parameter U-Net : {total_params:,} (~{total_params/1e6:.1f} Juta)")
print(f"Parameter Dapat Dilatih: {trainable_params} (WAJIB 0)")
assert trainable_params == 0, "[ERROR] Terdeteksi bobot U-Net yang tidak beku!"
print("[OK] Model U-Net 100% BEKU (frozen) sesuai Bab III L68!")

### Analisis Sel 2
Bobot U-Net berhasil diunduh dari repositori publik resmi Zenodo dan diverifikasi tidak ada satu pun parameter yang dapat diperbarui selama pelatihan klasifikasi (*frozen weights*).

In [ ]:
# Sel 3: Pemeriksaan Kuantitatif Pendukung pada Dataset RSUA (85 citra: 32 Normal, 53 Pneumonia)
from src.segmentation.qc import evaluate_rsua_dataset

rsua_root = PATHS["rsua_validated"]
print(f"Mengevaluasi batas paru pada dataset RSUA di: {rsua_root}")

rsua_qc = evaluate_rsua_dataset(unet_model, rsua_root, device=DEVICE)

print("\n=== HASIL QUALITY CONTROL U-NET PADA DATASET RSUA (85 CITRA) ===")
print(f"Status Kelulusan Keseluruhan : {rsua_qc['status']}")
print(f"Target Batas Ambang Proposal: DSC >= 0.85, IoU >= 0.75\n")

for cls, metrics in rsua_qc["summary_by_class"].items():
    pass_cls = (metrics['dice_mean'] >= 0.85) and (metrics['iou_mean'] >= 0.75)
    status_cls = "LULUS (PASS)" if pass_cls else "BELUM MEMENUHI (FAIL)"
    print(f"Kelas: {cls} (n = {metrics['n_samples']} citra) -> Status: {status_cls}")
    print(f"  - DSC (Dice) : {metrics['dice_mean']:.4f} +- {metrics['dice_std']:.4f} (Min: {metrics['dice_min']:.4f})")
    print(f"  - IoU (Paru) : {metrics['iou_mean']:.4f} +- {metrics['iou_std']:.4f} (Min: {metrics['iou_min']:.4f})")
    print(f"  - Rata2 r_paru: {metrics['r_paru_mean']:.4f}")

# Simpan rincian per citra
rsua_details_csv = PATHS["interim"] / "rsua_qc_details.csv"
rsua_qc["details_df"].to_csv(rsua_details_csv, index=False)
print(f"\nRincian evaluasi per citra tersimpan di: {rsua_details_csv}")

### Analisis Sel 3
Pemeriksaan kuantitatif pada dataset RSUA menguji ketepatan segmentasi model U-Net terhadap batas kontur paru dokter spesialis radiologi asli.

In [ ]:
# Sel 4: Pemeriksaan pada Citra Anak Kermany dkk. (>= 50 citra data uji)
import pandas as pd
from src.segmentation.qc import evaluate_kermany_subset_r_paru

split_csv = PATHS["interim"] / "split_patientwise.csv"
split_df = pd.read_csv(split_csv)
kermany_root = PATHS["kermany_chest_xray"]

print("Mengevaluasi sebaran luas paru r_paru pada subset data uji Kermany (25 Normal, 25 Pneumonia)...")
kermany_qc = evaluate_kermany_subset_r_paru(
    unet_model,
    split_df,
    kermany_root,
    device=DEVICE,
    n_per_class=25,
    seed=42,
)

print("\n=== HASIL SEBARAN LUAS PARU (r_paru) KERMANY ===")
print(f"Jumlah Sampel Diuji   : {kermany_qc['n_evaluated']} citra")
print(f"Rata-rata r_paru Normal    : {kermany_qc['mean_r_paru_normal']:.4f}")
print(f"Rata-rata r_paru Pneumonia : {kermany_qc['mean_r_paru_pneumonia']:.4f}")
print(f"Selisih |delta r_paru|     : {kermany_qc['delta_r_paru']:.4f} (Ambang: 0.05)")
print(f"Wajib Lapor ACR_norm?      : {kermany_qc['triggers_acr_norm']}")
print(f"Citra Gagal Ekstrem (<0.05): {kermany_qc['extreme_failure_count']} citra")

kermany_details_csv = PATHS["interim"] / "kermany_qc_subset_details.csv"
kermany_qc["details_df"].to_csv(kermany_details_csv, index=False)
print(f"Rincian tersimpan di: {kermany_details_csv}")

### Analisis Sel 4
Menilai apakah luas paru pada kelas Pneumonia berbeda signifikan dari kelas Normal yang dapat memengaruhi penilaian konsentrasi perhatian model (ACR).

In [ ]:
# Sel 5: Ekspor Visualisasi Overlay Kontur Paru pada Sampel Kermany & RSUA
import matplotlib.pyplot as plt
from scripts.export_overlay_for_manual_masks import run_overlay_export

overlay_dir = PATHS["outputs"] / "figures" / "lung_mask_overlays"
run_overlay_export(n_samples_per_class=5, output_dir=overlay_dir)

# Tampilkan visualisasi 4 sampel citra di notebook
exported_pngs = sorted(list(overlay_dir.glob("*.png")))[:4]
if exported_pngs:
    fig, axes = plt.subplots(1, len(exported_pngs), figsize=(16, 4))
    for ax, p in zip(axes, exported_pngs):
        im = Image.open(p)
        ax.imshow(im)
        ax.set_title(p.stem, fontsize=9)
        ax.axis("off")
    plt.tight_layout()
    plt.show()
    print(f"Visualisasi overlay kontur paru berhasil diekspor ke: {overlay_dir}")

### Analisis Sel 5
Garis hijau merepresentasikan batas wilayah anatomi paru hasil segmentasi U-Net yang telah dibersihkan secara morfologis (2 komponen terbesar dan pengisian lubang).

In [ ]:
# Sel 6: Susun dan Kunci Laporan Resmi Gerbang 3 (LUNG_MASK_QC_REPORT)
import json

qc_report = {
    "gate": "GERBANG_3_LUNG_MASK_TERVERIFIKASI",
    "unet_weights_source": "Zenodo 8393556 (Darwin + Montgomery + Shenzhen)",
    "unet_architecture": "smp.Unet(se_resnext101_32x4d)",
    "rsua_qc": {
        "status": rsua_qc["status"],
        "summary": rsua_qc["summary_by_class"],
    },
    "kermany_qc": {
        "mean_r_paru_normal": kermany_qc["mean_r_paru_normal"],
        "mean_r_paru_pneumonia": kermany_qc["mean_r_paru_pneumonia"],
        "delta_r_paru": kermany_qc["delta_r_paru"],
        "triggers_acr_norm": kermany_qc["triggers_acr_norm"],
        "extreme_failure_count": kermany_qc["extreme_failure_count"],
    },
}

qc_report_path = PATHS["interim"] / "lung_mask_qc_report.json"
with open(qc_report_path, "w", encoding="utf-8") as fh:
    json.dump(qc_report, fh, indent=2)

print(f"[SELESAI] Laporan QC Gerbang 3 berhasil dikunci di: {qc_report_path}")
print("Anda dapat melanjutkan ke notebook 03_preprocess_sanity_check.ipynb!")